# Qwen2.5-7B QLoRA Fine-tune → GGUF → Hugging Face → Ollama

Pipeline hoàn chỉnh:
`[Google Colab] ──(Quantize & Push)──> [HF Repo] ──(Pull)──> [Ollama Server (máy khác)]`

Bạn kết nối từ máy cá nhân tới Ollama server qua HTTP (`OLLAMA_HOST=http://<ip>:11434`).

---
**Yêu cầu:** GPU T4 / A100 trên Colab · Python ≥ 3.10 · `task_name` chọn `pathfinding` hoặc `course_rec`


## Cell 1 — Cài thư viện

In [1]:
# Chạy 1 lần. Restart runtime sau khi cài xong.
!pip -q install -U \
    transformers==4.45.0 \
    datasets \
    peft \
    trl==0.11.4 \
    accelerate \
    bitsandbytes \
    sentencepiece \
    huggingface_hub

# llama.cpp dùng ở Cell 9 — clone riêng khi đến bước đó


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 68.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 25.8 MB/s eta 0:00:00


## Cell 2 — Đăng nhập Hugging Face

> ⚠️ **KHÔNG** paste token thẳng vào code. Dùng Colab Secrets hoặc `notebook_login()`.

In [2]:
from huggingface_hub import notebook_login
# Mở popup nhập token HF của bạn (write permission).
# Hoặc dùng: login(token=userdata.get('HF_TOKEN'))  # Colab Secrets
notebook_login()


## Cell 3 — Cấu hình training

In [3]:
from dataclasses import dataclass
from pathlib import Path
import random, torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

@dataclass
class TrainConfig:
    model_name: str = "Qwen/Qwen2.5-7B-Instruct"
    # ── Chọn task ──────────────────────────────────────
    task_name: str = "pathfinding"   # hoặc "course_rec"
    # ── Output ─────────────────────────────────────────
    output_dir: str = "/content/qlora_out"
    merged_dir: str = "/content/merged_model"
    # ── Training ────────────────────────────────────────
    max_steps: int = 50          # tăng lên 300-500 khi có data thật
    train_batch_size: int = 1
    grad_accum_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    logging_steps: int = 5
    save_steps: int = 25
    eval_steps: int = 25
    max_seq_len: int = 2048
    # ── LoRA ────────────────────────────────────────────
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    # ── HF Push ─────────────────────────────────────────
    hf_repo_id: str = "huynh-dinh/career-it-advisor"  # đổi theo username

cfg = TrainConfig()
print(f"Task: {cfg.task_name}")
print(f"HF Repo: {cfg.hf_repo_id}")
print(f"CUDA available: {torch.cuda.is_available()}")


Task: pathfinding
HF Repo: huynh-dinh/career-it-advisor
CUDA available: True


## Cell 4 — Dataset

**Path JSONL thật (upload lên Colab trước):**
```
/content/data/ft_generator_pathfinding_train.jsonl
/content/data/ft_generator_pathfinding_val.jsonl
/content/data/ft_generator_course_rec_train.jsonl
/content/data/ft_generator_course_rec_val.jsonl
```
Nếu chưa có, fallback nhỏ sẽ được dùng để kiểm tra pipeline.

In [4]:
import json
from datasets import Dataset

DATA_DIR  = Path("/content/data")
TRAIN_PATH = DATA_DIR / f"ft_generator_{cfg.task_name}_train.jsonl"
VAL_PATH   = DATA_DIR / f"ft_generator_{cfg.task_name}_val.jsonl"


def read_jsonl_messages(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if "messages" not in obj:
                raise ValueError(f"Line {lineno} thiếu key 'messages' trong {path}")
            rows.append(obj)
    return rows


def build_fallback_rows(task: str) -> list[dict]:
    """Tiny dataset chỉ để test pipeline. KHÔNG dùng để train thật."""
    system = "Bạn là AI tư vấn hướng nghiệp IT. Chỉ dùng dữ liệu được cung cấp."
    if task == "course_rec":
        return [
            {"messages": [
                {"role": "system", "content": system},
                {"role": "user",   "content": "Câu hỏi: Khóa SQL nào phù hợp?\nDữ liệu Neo4j: {'found':true,'competency_name':'SQL','courses':[{'course_name':'SQL Cơ Bản','organization':'ABC','level':'Beginner'}]}"},
                {"role": "assistant", "content": "## Khóa học cho SQL\n\n**1. SQL Cơ Bản**\nTổ chức: ABC · Cấp độ: Beginner\n\nBạn muốn mình gợi ý lộ trình học SQL 30 ngày không?"},
            ]}
        ] * 4  # nhân lên để có ít nhất vài sample
    return [
        {"messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": "Câu hỏi: Làm Backend Developer cần học gì?\nDữ liệu Neo4j: {'found':true,'career_name':'Backend Developer','competencies':[{'name':'Python','kind':'ProgrammingLanguage','priority':1},{'name':'Docker','kind':'Tool','priority':2}]}"},
            {"role": "assistant", "content": "## Kỹ năng cần cho Backend Developer\n\n### ProgrammingLanguage\n- Python (ưu tiên 1)\n\n### Tool\n- Docker (ưu tiên 2)\n\nBạn muốn gợi ý khóa học cụ thể không?"},
        ]}
    ] * 4


if TRAIN_PATH.exists() and VAL_PATH.exists():
    train_rows = read_jsonl_messages(TRAIN_PATH)
    val_rows   = read_jsonl_messages(VAL_PATH)
    print(f"✅ Loaded real data — train: {len(train_rows)}, val: {len(val_rows)}")
else:
    train_rows = build_fallback_rows(cfg.task_name)
    val_rows   = build_fallback_rows(cfg.task_name)
    print("⚠️  Dùng fallback dataset (pipeline test only). Upload JSONL thật vào /content/data/")

train_ds = Dataset.from_list(train_rows)
val_ds   = Dataset.from_list(val_rows)
print(f"train_ds: {train_ds}")
print("Sample:", train_ds[0])


⚠️  Dùng fallback dataset (pipeline test only). Upload JSONL thật vào /content/data/
train_ds: Dataset({
    features: ['messages'],
    num_rows: 4
})
Sample: {'messages': [{'role': 'system', 'content': 'Bạn là AI tư vấn hướng nghiệp IT. Chỉ dùng dữ liệu được cung cấp.'}, {'role': 'user', 'content': "Câu hỏi: Làm Backend Developer cần học gì?\nDữ liệu Neo4j: {'found':true,'career_name':'Backend Developer','competencies':[{'name':'Python','kind':'ProgrammingLanguage','priority':1},{'name':'Docker','kind':'Tool','priority':2}]}"}, {'role': 'assistant', 'content': '## Kỹ năng cần cho Backend Developer\n\n### ProgrammingLanguage\n- Python (ưu tiên 1)\n\n### Tool\n- Docker (ưu tiên 2)\n\nBạn muốn gợi ý khóa học cụ thể không?'}]}


## Cell 5 — Load Model + Tokenizer (QLoRA 4-bit)

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training  # ← FIX: thiếu ở bản cũ

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # tránh warning với causal LM

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# ── FIX QUAN TRỌNG: bắt buộc gọi trước khi wrap PEFT ──
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

print("✅ Model + tokenizer ready")
print(f"   dtype: {next(model.parameters()).dtype}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model + tokenizer ready
   dtype: torch.float32


## Cell 6 — SFTTrainer

In [9]:
from trl import SFTConfig, SFTTrainer

IS_CUDA = torch.cuda.is_available()

sft_config = SFTConfig(
    output_dir=cfg.output_dir,
    max_steps=cfg.max_steps,
    per_device_train_batch_size=cfg.train_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    gradient_checkpointing=True,           # ← FIX: giảm VRAM ~40%
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    lr_scheduler_type="cosine",            # tốt hơn linear cho QLoRA
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    eval_steps=cfg.eval_steps,
    eval_strategy="steps",
    load_best_model_at_end=True,           # ← FIX: lấy checkpoint tốt nhất
    metric_for_best_model="eval_loss",
    bf16=IS_CUDA,                          # ← FIX: logic cũ bị ngược
    fp16=False,                            # không dùng fp16 khi có bf16
    max_seq_length=cfg.max_seq_len,
    packing=False,
    report_to="none",
)


def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )


trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    args=sft_config,
    formatting_func=formatting_func,
)

print("✅ Trainer ready")


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


✅ Trainer ready


## Cell 7 — Train

In [10]:
train_result = trainer.train()
print(train_result)

# Lưu adapter + tokenizer
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"✅ Adapter saved → {cfg.output_dir}")


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
25,0.000300,0.000409
50,0.000100,0.000183


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=50, training_loss=0.1671682691317983, metrics={'train_runtime': 1082.4136, 'train_samples_per_second': 0.37, 'train_steps_per_second': 0.046, 'total_flos': 1160513455718400.0, 'train_loss': 0.1671682691317983, 'epoch': 50.0})
✅ Adapter saved → /content/qlora_out


## Cell 8 — Kiểm tra nhanh sau train

In [15]:
import torch

# pipeline() không hỗ trợ PeftModelForCausalLM + QLoRA 4-bit → dùng generate() trực tiếp
model = trainer.model
model.eval()

if cfg.task_name == "course_rec":
    test_messages = [
        {"role": "system",  "content": "Bạn là AI tư vấn hướng nghiệp IT. Chỉ dùng dữ liệu được cung cấp."},
        {"role": "user",    "content": "Câu hỏi: khóa Python nào phù hợp cho người mới?\nDữ liệu Neo4j: {'found':true,'competency_name':'Python','courses':[{'course_name':'Python Cơ Bản','organization':'XYZ','level':'Beginner'}]}"},
    ]
else:
    test_messages = [
        {"role": "system",  "content": "Bạn là AI tư vấn hướng nghiệp IT. Chỉ dùng dữ liệu được cung cấp."},
        {"role": "user",    "content": "Câu hỏi: làm Data Analyst cần học gì?\nDữ liệu Neo4j: {'found':true,'career_name':'Data Analyst','competencies':[{'name':'SQL','kind':'ProgrammingLanguage','priority':1},{'name':'Power BI','kind':'Tool','priority':2}]}"},
    ]

prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt")
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.2,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
out = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("=== Model Output ===")
print(out)

RuntimeError: expected scalar type Float but found BFloat16

## Cell 9 — Merge LoRA Adapter → Full Model

> Bước này load model lên **CPU** để tránh OOM GPU khi merge. Cần ~14GB RAM (float16).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Dùng lại cfg.model_name và cfg.output_dir từ Cell 3
print("Đang load base model lên CPU...")
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)
tok = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)

print("Đang merge LoRA adapter...")
peft_model = PeftModel.from_pretrained(base_model, cfg.output_dir)
merged = peft_model.merge_and_unload()

merged.save_pretrained(cfg.merged_dir, safe_serialization=True)
tok.save_pretrained(cfg.merged_dir)
print(f"✅ Merged model saved → {cfg.merged_dir}")


## Cell 10 — Convert sang GGUF (llama.cpp)

Clone llama.cpp và build. Script `convert_hf_to_gguf.py` đã ổn định từ v0.3+.

In [ ]:
import os

TASK        = cfg.task_name          # pathfinding | course_rec
GGUF_F16    = f"/content/qwen25_{TASK}_f16.gguf"
GGUF_Q4     = f"/content/qwen25_{TASK}_q4km.gguf"  # file upload lên HF

# 1. Clone + build llama.cpp
if not os.path.isdir("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp

%cd /content/llama.cpp
!make -j$(nproc) llama-quantize 2>&1 | tail -5
!pip -q install -r requirements.txt
%cd /content

# 2. HF → GGUF F16
print("\n── Convert F16 ──")
!python /content/llama.cpp/convert_hf_to_gguf.py \
    {cfg.merged_dir} \
    --outfile {GGUF_F16} \
    --outtype f16

# 3. Quantize → Q4_K_M (cân bằng tốt nhất giữa chất lượng & kích thước)
print("\n── Quantize Q4_K_M ──")
!/content/llama.cpp/llama-quantize {GGUF_F16} {GGUF_Q4} q4_k_m

print(f"\n✅ GGUF sẵn sàng: {GGUF_Q4}")
!ls -lh /content/*.gguf


## Cell 11 — Push GGUF lên Hugging Face

> Đẩy cả **hai task** (pathfinding + course_rec) vào cùng 1 repo cho gọn.
> Sau đó máy chủ Ollama chỉ cần pull 1 repo.

In [ ]:
from huggingface_hub import HfApi

api     = HfApi()
repo_id = cfg.hf_repo_id

# Tạo repo nếu chưa có
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Upload file GGUF (Q4_K_M)
gguf_local = f"/content/qwen25_{cfg.task_name}_q4km.gguf"
gguf_remote = f"qwen25_{cfg.task_name}_q4km.gguf"

print(f"Uploading {gguf_local} → {repo_id}/{gguf_remote} ...")
api.upload_file(
    path_or_fileobj=gguf_local,
    path_in_repo=gguf_remote,
    repo_id=repo_id,
    repo_type="model",
)
print(f"✅ Done! https://huggingface.co/{repo_id}")

# Cũng push Modelfile tương ứng để tiện tham khảo
modelfile_content = f"""FROM ./{gguf_remote}

PARAMETER temperature 0.7
PARAMETER num_ctx 4096
PARAMETER repeat_penalty 1.1

SYSTEM \"\"\"Bạn là AI tư vấn {'khóa học IT' if cfg.task_name == 'course_rec' else 'hướng nghiệp IT'}. CHỈ dùng dữ liệu Neo4j được cung cấp.\"\"\"
"""
with open(f"/content/Modelfile.{cfg.task_name}", "w") as f:
    f.write(modelfile_content)

api.upload_file(
    path_or_fileobj=f"/content/Modelfile.{cfg.task_name}",
    path_in_repo=f"Modelfile.{cfg.task_name}",
    repo_id=repo_id,
    repo_type="model",
)
print(f"✅ Modelfile.{cfg.task_name} pushed")


## Cell 12 — Hướng dẫn cấu hình Ollama Server (máy khác)

Chạy các lệnh dưới đây **trên máy chủ** (không phải Colab).

```bash
# 1. Tải GGUF từ Hugging Face
huggingface-cli download huynh-dinh/career-it-advisor \
    qwen25_pathfinding_q4km.gguf   --local-dir ./models

huggingface-cli download huynh-dinh/career-it-advisor \
    qwen25_course_rec_q4km.gguf    --local-dir ./models

huggingface-cli download huynh-dinh/career-it-advisor \
    Modelfile.pathfinding          --local-dir ./models

huggingface-cli download huynh-dinh/career-it-advisor \
    Modelfile.course_rec           --local-dir ./models

# 2. Đăng ký model với Ollama
cd ./models
ollama create career-pathfinding -f Modelfile.pathfinding
ollama create career-course-rec  -f Modelfile.course_rec

# 3. Cho phép kết nối từ xa (máy khác trong mạng)
OLLAMA_HOST=0.0.0.0:11434 ollama serve
# Hoặc set biến môi trường vĩnh viễn:
# echo 'OLLAMA_HOST=0.0.0.0:11434' >> ~/.bashrc
```

---

### Kết nối từ máy cá nhân (Python)

```python
import requests

OLLAMA_SERVER = "http://<IP_MÁY_CHỦ>:11434"  # đổi IP thật

resp = requests.post(
    f"{OLLAMA_SERVER}/api/chat",
    json={
        "model": "career-pathfinding",
        "messages": [
            {"role": "user",
             "content": "Làm Backend Developer cần học gì?\nDữ liệu Neo4j: {...}"}
        ],
        "stream": False,
    }
)
print(resp.json()["message"]["content"])
```

### Kết nối từ app chatbot (OLLAMA_HOST env var)

```bash
# Trong .env của project backend:
OLLAMA_BASE_URL=http://<IP_MÁY_CHỦ>:11434
PATHFINDING_MODEL=career-pathfinding
COURSE_REC_MODEL=career-course-rec
```

## Ghi chú & Checklist

### Data thật (bước tiếp theo)

Mỗi dòng JSONL: `{"messages": [{role/system}, {role/user: câu hỏi + JSON Neo4j}, {role/assistant: trả lời grounded}]}`

- Tối thiểu **300–500 mẫu/task**, tốt hơn ≥ 800
- Chia train/val theo entity (không để cùng career/competency ở cả hai)
- Nhãn assistant lấy từ `format_pathfinding` / `format_course_rec`, chỉnh nhẹ cho tự nhiên

### Checklist fixes so với notebook cũ

| # | Vấn đề | Fix |
|---|--------|-----|
| 1 | Thiếu `prepare_model_for_kbit_training()` | ✅ Thêm vào Cell 5 |
| 2 | HF token hardcode trong code | ✅ Dùng `notebook_login()` |
| 3 | `fp16=True` khi có GPU (đáng lẽ dùng bf16) | ✅ Sửa logic |
| 4 | Thiếu `gradient_checkpointing=True` | ✅ Thêm |
| 5 | Thiếu `do_sample=True` cho temperature | ✅ Thêm |
| 6 | Không dùng `load_best_model_at_end` | ✅ Thêm |
| 7 | Cell 10 dùng `llama-quantize` nhưng binary build ra tên khác | ✅ Build `llama-quantize` đúng target |
| 8 | Modelfile không có `repeat_penalty` | ✅ Thêm 1.1 |
| 9 | Không có hướng dẫn kết nối remote Ollama | ✅ Cell 12 |
